In [12]:
import os

import matplotlib.pyplot as plt
import numpy as np
import gempy as gp
import matplotlib.pyplot as plt
import pandas as pd
import gempy_viewer as gpv
import HelperMethods as helper

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

path_to_data = r"C:\Users\mrcon\OneDrive\Data B\School\GP2\Internship\Mineye\GIS\Tharsis\Output\formationinputpoints.csv"
path_to_orientations = r"C:\Users\mrcon\OneDrive\Data B\School\GP2\Internship\Mineye\GIS\Tharsis\Output\orientations.csv"
path_to_topography = r"C:\Users\mrcon\OneDrive\Data B\School\GP2\Internship\Mineye\GIS\Tharsis\Output\topo.tif"
path_to_topography_cleaned = r"C:\Users\mrcon\OneDrive\Data B\School\GP2\Internship\Mineye\GIS\Tharsis\Output\topo_cleaned.tif"

In [10]:
points_df = pd.read_csv(path_to_data, encoding='latin1', engine='python')
points_df.keys()

Index(['X', 'Y', 'Z', 'formation_id', 'formation'], dtype='object')

In [11]:
orientations_df = pd.read_csv(path_to_orientations, encoding='latin1', engine='python')
orientations_df.keys()

Index(['X', 'Y', 'Z', 'azimuth', 'dip', 'polarity', 'formation',
       'formation_id'],
      dtype='object')

In [13]:
def smart_sample(sub_df, min_spacing):
    coords = sub_df[['X', 'Y']].values
    tree = cKDTree(coords)
    sampled_indices = []
    mask = np.ones(len(coords), dtype=bool)

    for i in range(len(coords)):
        if mask[i]:
            sampled_indices.append(i)
            neighbors = tree.query_ball_point(coords[i], r=min_spacing)
            mask[neighbors] = False
            mask[i] = True

    return sub_df.iloc[sampled_indices]

In [26]:
def sampling(df,BASE_SPACING=2500,ALPHA=0.50,plotting=False):
    formation_counts = df['formation'].value_counts()
    median_count = formation_counts.median()

    sampled_points = []

    for formation in df['formation'].unique():
        sub_df = df[df['formation'] == formation]
        count = len(sub_df)

        # Penalize formations with more data
        factor = (count / median_count) ** ALPHA
        effective_spacing = BASE_SPACING * factor

        print(f"Formation: {formation}, Points: {count}, Spacing: {effective_spacing:.2f}")

        sampled_df = smart_sample(sub_df, effective_spacing)
        sampled_points.append(sampled_df)

        if plotting == True:
            # Plot
            fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
            x_min, x_max = df['X'].min(), df['X'].max()
            y_min, y_max = df['Y'].min(), df['Y'].max()

            sc1 = axes[0].scatter(sub_df['X'], sub_df['Y'], c=sub_df['Z'], cmap='viridis', s=10)
            axes[0].set_title(f"Original: {formation}")
            axes[0].set_xlim(x_min, x_max)
            axes[0].set_ylim(y_min, y_max)
            axes[0].set_aspect('equal')
            plt.colorbar(sc1, ax=axes[0], label='Z')

            sc2 = axes[1].scatter(sampled_df['X'], sampled_df['Y'], c=sampled_df['Z'], cmap='viridis', s=10)
            axes[1].set_title(f"Sampled: {formation}")
            axes[1].set_xlim(x_min, x_max)
            axes[1].set_ylim(y_min, y_max)
            axes[1].set_aspect('equal')
            plt.colorbar(sc2, ax=axes[1], label='Z')

            for ax in axes:
                ax.set_xlabel('X')
                ax.set_ylabel('Y')

            plt.tight_layout()
            plt.show()

    # Combine all sampled
    sampled_df_all = pd.concat(sampled_points, ignore_index=True)

    return sampled_df_all

In [29]:
sampled_points_df       = sampling(points_df,plotting=False)
sampled_orientations_df = sampling(orientations_df, BASE_SPACING=5500, ALPHA=1.90)

Formation: Upper Devonian Siliciclastics, Points: 18, Spacing: 2500.00
Formation: Visean Shales, Points: 16, Spacing: 2357.02
Formation: Tournaisian Plutonites, Points: 296, Spacing: 10137.94
Formation: Upper Carboniferous Volcanics, Points: 45, Spacing: 3952.85
Formation: Mid Carboniferous Shales, Points: 13, Spacing: 2124.59
Formation: Upper Carboniferous Volcanics, Points: 1, Spacing: 5500.00
Formation: Upper Devonian Siliciclastics, Points: 3, Spacing: 44349.94
Formation: Tournaisian Plutonites, Points: 1, Spacing: 5500.00
Formation: Mid Carboniferous Shales, Points: 1, Spacing: 5500.00


In [33]:
# Create GemPy model using the sampled/reduced points
print(f"Total sampled points: {len(sampled_points_df)}")
print(f"Formation distribution in sampled data:")
print(sampled_points_df['formation'].value_counts())

# Save the sampled data to CSV files for GemPy
sampled_points_path = "sampled_formationinputpoints.csv"
sampled_points_df.to_csv(sampled_points_path, index=False)

Total sampled points: 19
Formation distribution in sampled data:
Upper Carboniferous Volcanics    6
Tournaisian Plutonites           5
Upper Devonian Siliciclastics    3
Visean Shales                    3
Mid Carboniferous Shales         2
Name: formation, dtype: int64


In [34]:
# Create GemPy model using the sampled/reduced orientations
print(f"Total sampled orientations: {len(sampled_orientations_df)}")
print(f"Formation distribution in sampled data:")
print(sampled_orientations_df['formation'].value_counts())

# Save the sampled data to CSV files for GemPy
sampled_orientations_path = "sampled_orientationinputpoints.csv"
sampled_orientations_df.to_csv(sampled_orientations_path, index=False)

Total sampled orientations: 4
Formation distribution in sampled data:
Upper Carboniferous Volcanics    1
Upper Devonian Siliciclastics    1
Tournaisian Plutonites           1
Mid Carboniferous Shales         1
Name: formation, dtype: int64


In [36]:
# Calculate combinations for picking 3 points randomly from each formation
import math
from math import comb

print("Calculating combinations for picking 3 points from each formation:")
print("=" * 60)

# Get formation counts
formation_counts = sampled_points_df['formation'].value_counts().sort_index()
print("Formation point counts:")
for formation, count in formation_counts.items():
    print(f"  {formation}: {count} points")

print("\n" + "=" * 60)
print("Combinations analysis:")
print("=" * 60)

total_combinations = 1
valid_formations = 0
formations_with_insufficient_points = []

for formation, count in formation_counts.items():
    if count >= 3:
        # Calculate C(n,3) = n! / (3! * (n-3)!)
        combinations = comb(count, 3)
        print(f"{formation:30s}: C({count:3d}, 3) = {combinations:,}")
        total_combinations *= combinations
        valid_formations += 1
    else:
        print(f"{formation:30s}: Only {count} points - insufficient for 3-point selection")
        formations_with_insufficient_points.append(formation)

print("\n" + "=" * 60)
print("SUMMARY:")
print("=" * 60)
print(f"Total formations: {len(formation_counts)}")
print(f"Formations with ≥3 points: {valid_formations}")
print(f"Formations with <3 points: {len(formations_with_insufficient_points)}")

if formations_with_insufficient_points:
    print(f"\nFormations with insufficient points:")
    for formation in formations_with_insufficient_points:
        print(f"  - {formation}")

if valid_formations > 0:
    print(f"\nTotal possible combinations:")
    print(f"  {total_combinations:,}")
    
    # Convert to scientific notation for large numbers
    if total_combinations > 1e6:
        print(f"  {total_combinations:.2e} (scientific notation)")
        
    # Estimate computation time (rough calculation)
    print(f"\nRough estimates:")
    print(f"  - At 1,000 combinations/second: {total_combinations/1000:.1f} seconds")
    print(f"  - At 10,000 combinations/second: {total_combinations/10000:.1f} seconds")
    
    if total_combinations > 1e9:
        print(f"  ⚠️  This is a very large number - consider sampling strategies!")
else:
    print(f"\n❌ No formations have enough points for 3-point selection")

print("=" * 60)

Calculating combinations for picking 3 points from each formation:
Formation point counts:
  Mid Carboniferous Shales: 2 points
  Tournaisian Plutonites: 5 points
  Upper Carboniferous Volcanics: 6 points
  Upper Devonian Siliciclastics: 3 points
  Visean Shales: 3 points

Combinations analysis:
Mid Carboniferous Shales      : Only 2 points - insufficient for 3-point selection
Tournaisian Plutonites        : C(  5, 3) = 10
Upper Carboniferous Volcanics : C(  6, 3) = 20
Upper Devonian Siliciclastics : C(  3, 3) = 1
Visean Shales                 : C(  3, 3) = 1

SUMMARY:
Total formations: 5
Formations with ≥3 points: 4
Formations with <3 points: 1

Formations with insufficient points:
  - Mid Carboniferous Shales

Total possible combinations:
  200

Rough estimates:
  - At 1,000 combinations/second: 0.2 seconds
  - At 10,000 combinations/second: 0.0 seconds
